- Data statistics
- Data quality issues
- Visualizations
- Initial preprocessing decisions

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image

# Load data from local path
dataset_path = Path('csiro-biomass')
train_df = pd.read_csv(dataset_path / 'train.csv')
test_df = pd.read_csv(dataset_path / 'test.csv')
sample_submission = pd.read_csv(dataset_path / 'sample_submission.csv')

print(f"Training set: {train_df.shape[0]} rows, {train_df.shape[1]} columns") 
print(f"Test set: {test_df.shape[0]} rows, {test_df.shape[1]} columns")
print(f"Sample submission set: {sample_submission.shape[0]} rows, {sample_submission.shape[1]} columns")
print(f"\nColumns: {list(train_df.columns)}")
train_df.head(10)

## Load and Explore Metadata

In [ ]:
# Data types and missing values
print("Data Types:")
print(train_df.dtypes)
print("\n" + "="*50)
print("\nMissing Values:")
print(train_df.isnull().sum())

In [ ]:
# Test data set info
test_df.head()

## Target Variable Distributions
The 5 biomass targets:
- Dry clover biomass
- Dry dead material  
- Dry green vegetation
- Total dry biomass
- Green dry matter


In [ ]:
# Count of each target type
print("Target Type Counts:")
print(train_df['target_name'].value_counts())
print()

# Distribution of each target type
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
target_types = train_df['target_name'].unique()

for i, target in enumerate(target_types):
    subset = train_df[train_df['target_name'] == target]['target']
    axes[i].hist(subset.dropna(), bins=30, edgecolor='black', alpha=0.7, color='steelblue')
    axes[i].set_title(f'{target}', fontsize=10, fontweight='bold')
    axes[i].set_xlabel('Value (g)')
    axes[i].set_ylabel('Frequency')
    axes[i].axvline(subset.mean(), color='red', linestyle='--', linewidth=1)

plt.tight_layout()
plt.suptitle('Distribution of Target Variables', fontsize=14, fontweight='bold', y=1.05)
plt.show()

## Sample Images

In [ ]:
from PIL import Image
import os

# Get list of image files
image_files = train_df['image_path'].apply(lambda x: dataset_path / x)

# Check sizes of all images
image_sizes = {}
image_modes = {}
for img_path in image_files:
    with Image.open(img_path) as img:
        image_sizes[img.size] = image_sizes.get(img.size, 0) + 1
        image_modes[img.mode] = image_modes.get(img.mode, 0) + 1
print(image_sizes)
print(image_modes)

# Display sample images
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, img_path in enumerate(image_files[:6]):
    img = Image.open(img_path)
    axes[i].imshow(img)
    axes[i].set_title(img_path.name, fontsize=10)
    axes[i].axis('off')

plt.tight_layout()
plt.suptitle('Sample Training Images', fontsize=14, fontweight='bold', y=1.02)
plt.show()

# Image size info
sample_img = Image.open(image_files[0])
print(f"\nImage size: {sample_img.size}")
print(f"Image mode: {sample_img.mode}")

## Initial Preprocessing

In [ ]:
# Image preprocessing with Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

IMAGE_SIZE = 224

# Calculate actual mean and variance from training images
# print("Calculating dataset mean and variance...")
# pixel_sum = tf.zeros([3], dtype=tf.float64)
# pixel_sq_sum = tf.zeros([3], dtype=tf.float64)
# pixel_count = 0

# for img_path in train_df['image_path']:
#     full_path = dataset_path / img_path
#     img = tf.io.read_file(str(full_path))
#     img = tf.image.decode_jpeg(img, channels=3)
#     img = tf.cast(img, tf.float64) / 255.0  # Scale to [0, 1]
    
#     pixel_sum += tf.reduce_sum(img, axis=[0, 1])
#     pixel_sq_sum += tf.reduce_sum(tf.square(img), axis=[0, 1])
#     pixel_count += img.shape[0] * img.shape[1]

# Calculate mean and variance
#dataset_mean = (pixel_sum / pixel_count).numpy().tolist() # ['0.4379', '0.5048', '0.3002']
dataset_mean = ['0.4379', '0.5048', '0.3002']
#dataset_variance = ((pixel_sq_sum / pixel_count) - tf.square(pixel_sum / pixel_count)).numpy().tolist() # ['0.0633', '0.0626', '0.0576']
dataset_variance = ['0.0633', '0.0626', '0.0576']

print(f"Dataset Mean (RGB): {[f'{m:.4f}' for m in dataset_mean]}")
print(f"Dataset Variance (RGB): {[f'{v:.4f}' for v in dataset_variance]}")

# Training augmentation layers with actual dataset statistics
train_augmentation = keras.Sequential([
    layers.Resizing(IMAGE_SIZE, IMAGE_SIZE),
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(factor=15/360),
    layers.RandomBrightness(factor=0.2),
    layers.RandomContrast(factor=0.2),
    layers.Rescaling(1./255),
    layers.Normalization(mean=dataset_mean, variance=dataset_variance)
])

# Validation preprocessing (no augmentation)
val_preprocessing = keras.Sequential([
    layers.Resizing(IMAGE_SIZE, IMAGE_SIZE),
    layers.Rescaling(1./255),
    layers.Normalization(mean=dataset_mean, variance=dataset_variance)
])

# Test preprocessing on sample image
sample_path = dataset_path / train_df['image_path'].iloc[0]
sample_img = tf.io.read_file(str(sample_path))
sample_img = tf.image.decode_jpeg(sample_img, channels=3)
sample_img = tf.cast(sample_img, tf.float32)

# Apply training augmentation (also builds the model)
transformed_img = train_augmentation(tf.expand_dims(sample_img, 0))
print(f"\nTransformed image shape: {transformed_img.shape}")
print(f"Pixel value range: [{tf.reduce_min(transformed_img):.2f}, {tf.reduce_max(transformed_img):.2f}]")

# Build validation preprocessing model too
_ = val_preprocessing(tf.expand_dims(sample_img, 0))
print("Validation preprocessing built successfully")

In [ ]:
# View sample image: original vs transformed
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
target_columns = train_df['target_name'].unique()
# Original image
print(train_data[target_columns].iloc[0])
sample_path = dataset_path / train_df['image_path'].iloc[0]
original_img = Image.open(sample_path)
axes[0].imshow(original_img)
axes[0].set_title(f'Original Image\nSize: {original_img.size}', fontsize=12)
axes[0].axis('off')

# Transformed image (denormalize for visualization)
sample_tensor = tf.io.read_file(str(sample_path))
sample_tensor = tf.image.decode_jpeg(sample_tensor, channels=3)
sample_tensor = tf.cast(sample_tensor, tf.float32)
augmented = train_augmentation(tf.expand_dims(sample_tensor, 0))[0]

# Denormalize: x * std + mean
std = tf.sqrt(dataset_variance)
denorm_img = augmented * std + dataset_mean
denorm_img = tf.clip_by_value(denorm_img, 0, 1)

axes[1].imshow(augmented.numpy())
axes[1].set_title(f'After Augmentation\nSize: {IMAGE_SIZE}x{IMAGE_SIZE}', fontsize=12)
axes[1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split

# Pivot data so each image has one row with all 5 targets
train_pivoted = train_df.pivot_table(
    index=['image_path', 'Sampling_Date', 'State', 'Species', 'Pre_GSHH_NDVI', 'Height_Ave_cm'],
    columns='target_name',
    values='target'
).reset_index()

print(f"Pivoted data shape: {train_pivoted.shape}")
print(f"Unique images: {train_pivoted['image_path'].nunique()}")

# Train/Validation split (stratified by State to ensure balanced representation)
train_data, val_data = train_test_split(
    train_pivoted, 
    test_size=0.2, 
    random_state=42, 
    stratify=train_pivoted['State']
)

train_pivoted.head()

In [ ]:
# Create tf.data.Dataset for training and validation

# Target column names (from pivoted data)
target_columns = train_df['target_name'].unique()

def create_dataset(df, preprocessing_fn, batch_size=32, shuffle=True):
    """Create a tf.data.Dataset from dataframe with image paths and targets."""
    
    # Get image paths and targets
    image_paths = [str(dataset_path / p) for p in df['image_path'].values]
    targets = df[target_columns].values.astype('float32')
    
    def load_and_preprocess(path, target):
        # Load image
        img = tf.io.read_file(path)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.cast(img, tf.float32)
        # Add batch dimension for Keras layers, then remove it
        img = tf.expand_dims(img, 0)
        img = preprocessing_fn(img)
        img = tf.squeeze(img, 0)
        return img, target
    
    # Create dataset from tensors
    dataset = tf.data.Dataset.from_tensor_slices((image_paths, targets))
    
    if shuffle:
        dataset = dataset.shuffle(buffer_size=len(df))
    
    dataset = dataset.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    
    return dataset

# Create train and validation datasets
BATCH_SIZE = 16

train_dataset = create_dataset(train_data, train_augmentation, batch_size=BATCH_SIZE, shuffle=True)
val_dataset = create_dataset(val_data, val_preprocessing, batch_size=BATCH_SIZE, shuffle=True)



In [ ]:
# Verify train dataset
for images, targets in train_dataset.take(1):
    print(f"Image batch shape: {images.shape}")
    print(f"Target batch shape: {targets.shape}")
    plt.imshow(images[0].numpy())
    print(f"\nTarget columns: {list(target_columns)}")
    print(f"Sample targets (first row): {targets[0].numpy()}")

print(f"\nTrain dataset: {len(train_data)} images")


In [ ]:
# Verify validation dataset
for images, targets in val_dataset.take(1):
    print(f"Image batch shape: {images.shape}")
    print(f"Target batch shape: {targets.shape}")
    plt.imshow(images[0].numpy())
    print(f"\nTarget columns: {list(target_columns)}")
    print(f"Sample targets (first row): {targets[0].numpy()}")

print(f"Validation dataset: {len(val_data)} images")